In [10]:
import pandas as pd
import re

# Function to convert time string to seconds
def time_to_seconds(time_str):
    h, m, s = map(float, re.findall(r'\[?(\d+):(\d+):(\d+\.\d+)\]?', time_str)[0])
    return h * 3600 + m * 60 + s

# Read the log file
with open('data/elapsed_times.log', 'r') as file:
    lines = file.readlines()

# Prepare data
data = []
for line in lines:
    line = line.strip()

    # Extract id using regex
    id_match = re.search(r'\[([^\]]+)\]', line)
    if id_match:
        id = id_match.group(1)
    else:
        id = None  # Handle unexpected lines if necessary

    # Extract elapsed time
    elapsed_time_str = re.search(r'Elapsed time \[([^\]]+)\]', line).group(1)
    elapsed = time_to_seconds(elapsed_time_str)

    # Extract cores
    cores_str = re.search(r'\[(\d+)\] cores', line).group(1)
    cores = int(cores_str)

    data.append({"id": id, "elapsed": elapsed, "cores": cores})

# Create DataFrame
df = pd.DataFrame(data)

# Read valid_dataset.csv
valid_df = pd.read_csv('data/valid_dataset.csv', sep=';')

# Add 'valid' column
df['valid'] = df['id'].isin(valid_df['id'])

# Write to CSV
df.to_csv("results/elapsed_times.csv", index=False, sep=';')

valid_count = df['valid'].sum()
print(f"Valid : {valid_count}")

df.head(1001)


Valid : 744


,id,elapsed,cores,valid
0,37dd92ca7ddadf95a5a17a767eb40f0d37e2a28fdb6e0a...,296.402,48,False
1,5c4ebd5a6bb75486ffaa7aecc4fe1b0568938f924a35ab...,252.348,48,False
2,d9da52a92f272e67063ad3f4c0f003565c05c9dfaadc4a...,250.358,48,True
3,76e40d72b9325e876083f6da7512419ba4d73a1aa6def6...,266.366,48,True
4,a0f0593cd25b376403e53368ccfd92668aae08f5ecdb63...,248.349,48,True
...,...,...,...,...
996,cc3ea6fc8b5ba505aeddb8a3f9762e1040a42ee84fdcba...,244.338,48,True
997,efabab0d166696787a44c4ce7504d68cbab410cde6a471...,236.327,48,False
998,b33c7a0cbf37684ddd1479e6a25c5ce4c15d257c22d622...,238.336,48,True
999,3d973ab36aa16fbedb41fa529f88b81c736803f9d035d4...,270.355,48,False


In [11]:
# Mean elapsed time for all lines
mean_all = df['elapsed'].mean()

# Mean elapsed time for valid lines
mean_valid = df[df['valid']]['elapsed'].mean()

# Mean elapsed time for non-valid lines
mean_non_valid = df[~df['valid']]['elapsed'].mean()

print(f"Mean elapsed time (all): {mean_all:.2f} seconds")
print(f"Mean elapsed time (valid): {mean_valid:.2f} seconds")
print(f"Mean elapsed time (invalid): {mean_non_valid:.2f} seconds")


Mean elapsed time (all): 255.17 seconds
Mean elapsed time (valid): 253.89 seconds
Mean elapsed time (invalid): 258.88 seconds


In [15]:
# Total elapsed time for all configurations
total_all_seconds = df['elapsed'].sum()

# Total elapsed time for valid configurations
total_valid_seconds = df[df['valid']]['elapsed'].sum()

# Total elapsed time for non-valid configurations
total_non_valid_seconds = df[~df['valid']]['elapsed'].sum()

# Function to convert seconds to HH:mm:ss
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

# Format the total times
formatted_all = format_time(total_all_seconds)
formatted_valid = format_time(total_valid_seconds)
formatted_non_valid = format_time(total_non_valid_seconds)

print(f"Total execution time (all configs): {formatted_all}")
print(f"Total execution time (valid configs): {formatted_valid}")
print(f"Total execution time (non-valid configs): {formatted_non_valid} ({(total_non_valid_seconds / total_all_seconds) * 100} %)")

Total execution time (all configs): 70:57:04
Total execution time (valid configs): 52:28:11
Total execution time (non-valid configs): 18:28:52 (26.047782451000145 %)
26.047782451000145
